In [4]:
# Import necessary libraries
import cv2
import numpy as np

from shapely import Polygon

SPEED_THRESHOLD = 2
AREA_THRESHOLD = 2000

# Read in a video file
vidReader = cv2.VideoCapture('visiontraffic.avi')

# Skip first still frames
for i in range(90):
    ret, frame = vidReader.read()

# Initialize optical flow
prevGray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# Get the frame width and height
frame_width = int(vidReader.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vidReader.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create a VideoWriter object
out = cv2.VideoWriter('output.avi', cv2.VideoWriter_fourcc(*'XVID'), 20.0, (frame_width, frame_height))

detections = []
current_i = 0

def get_detection(contour: np.ndarray) -> dict:
    try:
        poly = Polygon(contour)
    except:
        return None
    for existing in detections:
        if existing["ok"] == False:
            continue
        existing_poly = Polygon(existing["contour"])
        intersection = poly.intersection(existing_poly)
        union = poly.union(existing_poly)
        if intersection.area / union.area > 0.3:
            return existing
    return None

while vidReader.isOpened():
    ret, frameRGB = vidReader.read()
    if not ret:
        break
    frameGray = cv2.cvtColor(frameRGB, cv2.COLOR_BGR2GRAY)

    # Estimate optical flow using Farneback method
    flow = cv2.calcOpticalFlowFarneback(prevGray, frameGray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

    # Calculate speed and direction from Vx and Vy
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    # Remove all measurements for speeds lower than SPEED_THRESHOLD
    mask = mag > SPEED_THRESHOLD
    filtdir = np.zeros_like(ang)
    filtdir[mask] = ang[mask]

    # Calculate region statistics for thresholded image
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Analyze found regions
    annotations = []

    for contour in contours:
        contour_2d = contour.reshape(-1, 2)
        area = cv2.contourArea(contour)
        existing = get_detection(contour_2d)
        if area > AREA_THRESHOLD:
            bb = cv2.boundingRect(contour)
            x, y, w, h = bb

            dirbb = filtdir[y:y+h, x:x+w]
            spdbb = mag[y:y+h, x:x+w]
            det_dir = np.mean(dirbb[dirbb != 0])
            det_spd = np.mean(spdbb[spdbb > 2])
            cx, cy = np.mean(contour, axis=0)[0]

            detection = {
                'ok': True,
                'bb': bb,
                'dir': det_dir,
                'spd': det_spd,
                'contour': contour_2d,
                'cc': (cx, cy),
                'lbl': area
            }
            
            if existing is None:
                detection["i"] = current_i
                current_i += 1
                detections.append(detection)
                result = detection
            else:
                i = existing["i"]
                detections[i] |= detection
                result = detections[i]

            annotations.append(result)
        elif existing is not None:
            existing["ok"] = False


    # Draw annotations (bounding boxes with the segment area)
    ann = frameRGB.copy()
    for det in annotations:
        x, y, w, h = det['bb']
        cv2.rectangle(ann, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(ann, str(int(det['i'])), (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Draw the average direction vector
    for det in annotations:
        x1, y1 = det['cc']
        x2 = int(x1 + 5 * det['spd'] * np.cos(det['dir']))
        y2 = int(y1 + 5 * det['spd'] * np.sin(det['dir']))
        cv2.arrowedLine(ann, (int(x1), int(y1)), (x2, y2), (0, 0, 255), 3)

    # Write the annotated frame to the output video
    out.write(ann)

    # Update previous frame
    prevGray = frameGray

vidReader.release()
out.release()